# Pro-Level SVC Notebook: Banking, HR, and Health Sector

This notebook teaches **Support Vector Classifier (SVC)** using three sector-based examples:

1. **Banking**: Loan approval prediction  
2. **HR**: Employee attrition prediction  
3. **Health**: Diabetes prediction

It includes:

- CSV loading
- Feature scaling with `StandardScaler`
- SVC model training
- Accuracy, confusion matrix, and classification report
- Decision boundary visualization
- Kernel comparison: Linear, RBF, Polynomial
- GridSearchCV hyperparameter tuning
- Final prediction on new data

> Teaching reminder: **SVC is for classification**. It predicts classes such as approved/rejected, leave/stay, positive/negative.


## 1. Install / Import Libraries
Run this notebook in Jupyter, Anaconda, VS Code, or Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

print('Libraries imported successfully')

## 2. Load CSV Files
Make sure the CSV files are in the same folder as this notebook.

In [ ]:
bank_df = pd.read_csv('banking_loan_data_pro.csv')
hr_df = pd.read_csv('hr_attrition_data_pro.csv')
health_df = pd.read_csv('health_diabetes_data_pro.csv')

print('Banking Data')
display(bank_df.head())

print('HR Data')
display(hr_df.head())

print('Health Data')
display(health_df.head())

## 3. Helper Function: Train and Evaluate SVC
This function trains an SVC model using a pipeline:

`StandardScaler → SVC`

Scaling is very important because SVM is distance-based.

In [ ]:
def train_evaluate_svc(df, feature_cols, target_col, title, kernel='linear'):
    X = df[feature_cols]
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(kernel=kernel, probability=True, random_state=42))
    ])
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print('='*70)
    print(title)
    print('='*70)
    print('Kernel:', kernel)
    print('Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
    print('
Classification Report:
')
    print(classification_report(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f'{title} - Confusion Matrix')
    plt.show()
    
    return model

# Part A: Banking Sector — Loan Approval Prediction

**Target variable:** `approved`

- `0` = Loan rejected  
- `1` = Loan approved

In [ ]:
bank_features = ['income', 'credit_score', 'loan_amount']
bank_target = 'approved'

bank_model = train_evaluate_svc(
    bank_df,
    bank_features,
    bank_target,
    'Banking SVC: Loan Approval',
    kernel='linear'
)

# Part B: HR Sector — Employee Attrition Prediction

**Target variable:** `attrition`

- `0` = Employee may stay  
- `1` = Employee may leave

In [ ]:
hr_features = ['age', 'monthly_income', 'job_satisfaction', 'overtime']
hr_target = 'attrition'

hr_model = train_evaluate_svc(
    hr_df,
    hr_features,
    hr_target,
    'HR SVC: Employee Attrition',
    kernel='linear'
)

# Part C: Health Sector — Diabetes Prediction

**Target variable:** `diabetes`

- `0` = Diabetes negative  
- `1` = Diabetes positive

In [ ]:
health_features = ['glucose', 'blood_pressure', 'bmi', 'age']
health_target = 'diabetes'

health_model = train_evaluate_svc(
    health_df,
    health_features,
    health_target,
    'Health SVC: Diabetes Prediction',
    kernel='linear'
)

# 4. Pro Visualization: SVC Decision Boundary

A true SVC boundary can be visualized easily with **2 features only**.

For banking, we use:

- `income`
- `credit_score`

The line/curve separates the two classes.

In [ ]:
def plot_decision_boundary(df, x_col, y_col, target_col, kernel='linear', title='SVC Decision Boundary'):
    X = df[[x_col, y_col]].values
    y = df[target_col].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    model = SVC(kernel=kernel, C=1.0, gamma='scale')
    model.fit(X_scaled, y)
    
    x_min, x_max = X_scaled[:, 0].min() - 0.8, X_scaled[:, 0].max() + 0.8
    y_min, y_max = X_scaled[:, 1].min() - 0.8, X_scaled[:, 1].max() + 0.8
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.25)
    plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, edgecolors='k', s=80)
    plt.xlabel(f'{x_col} (scaled)')
    plt.ylabel(f'{y_col} (scaled)')
    plt.title(f'{title} | Kernel = {kernel}')
    plt.show()

plot_decision_boundary(
    bank_df,
    'income',
    'credit_score',
    'approved',
    kernel='linear',
    title='Banking Loan Approval Boundary'
)

## 5. Compare SVC Kernels

SVC supports different kernels:

- `linear`: straight boundary
- `rbf`: curved/non-linear boundary
- `poly`: polynomial boundary

This is useful for teaching how SVM separates data.

In [ ]:
for kernel in ['linear', 'rbf', 'poly']:
    plot_decision_boundary(
        bank_df,
        'income',
        'credit_score',
        'approved',
        kernel=kernel,
        title='Banking Loan Approval Boundary'
    )

# 6. GridSearchCV for Best SVC Parameters

GridSearchCV tests multiple values of:

- `C`: regularization strength
- `kernel`: type of boundary
- `gamma`: influence of each training point

In [ ]:
X = bank_df[bank_features]
y = bank_df[bank_target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42))
])

param_grid = {
    'svc__kernel': ['linear', 'rbf', 'poly'],
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 'auto']
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print('Best Parameters:', grid.best_params_)
print('Best CV Accuracy:', round(grid.best_score_ * 100, 2), '%')

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print('Test Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title('Best Banking SVC Model - Confusion Matrix')
plt.show()

# 7. Predict New Real-Life Examples

Now we use the trained models on new unseen records.

In [ ]:
# Banking prediction
new_customer = pd.DataFrame({
    'income': [75000],
    'credit_score': [740],
    'loan_amount': [130000]
})

prediction = bank_model.predict(new_customer)[0]
print('Banking Prediction:', 'Loan Approved' if prediction == 1 else 'Loan Rejected')

# HR prediction
new_employee = pd.DataFrame({
    'age': [29],
    'monthly_income': [32000],
    'job_satisfaction': [1],
    'overtime': [1]
})

prediction = hr_model.predict(new_employee)[0]
print('HR Prediction:', 'Employee May Leave' if prediction == 1 else 'Employee May Stay')

# Health prediction
new_patient = pd.DataFrame({
    'glucose': [165],
    'blood_pressure': [92],
    'bmi': [34.2],
    'age': [55]
})

prediction = health_model.predict(new_patient)[0]
print('Health Prediction:', 'Diabetes Positive' if prediction == 1 else 'Diabetes Negative')

# 8. Lecture Summary

## SVC Meaning
SVC means **Support Vector Classifier**. It is used for classification problems.

## Main Idea
SVC tries to find the best boundary between classes.

## Important Terms

- **Hyperplane**: Decision boundary that separates classes
- **Support Vectors**: Important data points near the boundary
- **Margin**: Distance between the boundary and support vectors
- **Kernel**: Function used to create linear or non-linear boundaries
- **C**: Controls strictness of classification
- **Gamma**: Controls how much influence one data point has

## Sector Examples

| Sector | Problem | Target |
|---|---|---|
| Banking | Loan approval | Approved / Rejected |
| HR | Employee attrition | Leave / Stay |
| Health | Diabetes prediction | Positive / Negative |
